##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# GitHub Repository Governance with the Gemini API

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/github-api/GitHub_API_with_Gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/>
</a>

## Overview

This notebook shows you how to combine the [GitHub REST API](https://docs.github.com/en/rest) with the
[Gemini API](https://ai.google.dev/gemini-api/docs) to build AI-assisted repository governance tools.

You will work through four practical scenarios:

1. **Fetch repository metadata** — retrieve repo info from GitHub and ask Gemini to write an
   executive summary.
2. **Analyze file compliance (structured output)** — fetch a file's raw content and use Gemini
   with a `response_schema` to extract structured compliance findings.
3. **PR review assistant** — fetch a pull request diff and ask Gemini to produce an actionable
   code review comment.
4. **Agent schema validator** — validate a YAML agent definition against a governance schema and
   generate a compliance report.

These patterns are directly applicable to projects that use a GitHub repository to govern AI agent
prompts with compliance schemas, IP registries, and CI/CD validation pipelines.

### Prerequisites

- A Gemini API key stored in the environment variable `GEMINI_API_KEY`.
- Optionally, a GitHub personal access token in `GITHUB_TOKEN` (required for private repos and
  to avoid rate-limiting on public repos).

## Setup

### Install dependencies

In [ ]:
%pip install -U -q 'google-genai>=1.0.0' requests

### Import libraries and configure clients

In [ ]:
import json
import os

import requests
from google import genai
from google.genai import types

### Configure your API key

Your Gemini API key must be available as the environment variable `GEMINI_API_KEY`. If you are
running in Colab, store it as a Colab Secret with that name and load it with the snippet below.
See the [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb)
quickstart for detailed instructions.

In [ ]:
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

### Configure your GitHub token (optional)

A GitHub personal access token is not required for public repositories, but it raises the
unauthenticated rate limit from 60 to 5,000 requests per hour and is required for private
repositories. Store the token as the environment variable `GITHUB_TOKEN`.

In [ ]:
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

github_headers = {"Accept": "application/vnd.github+json"}
if GITHUB_TOKEN:
    github_headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

### Select a model

In [ ]:
MODEL_ID = "gemini-2.5-flash"  # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3-flash-preview"] {"allow-input":true, isTemplate: true}

## 1. Fetch repository metadata and generate an executive summary

The GitHub REST API exposes rich metadata for any public repository at
`/repos/{owner}/{repo}`. Fetch that JSON and pass it to Gemini to produce a concise
executive summary — useful for onboarding documents, dashboards, or governance reports.

In [ ]:
OWNER = "google-gemini"
REPO = "cookbook"

repo_response = requests.get(
    f"https://api.github.com/repos/{OWNER}/{REPO}",
    headers=github_headers,
)
repo_response.raise_for_status()
repo_data = repo_response.json()

# Print a subset of the fields to confirm the fetch succeeded.
summary_fields = {
    k: repo_data[k]
    for k in (
        "full_name",
        "description",
        "stargazers_count",
        "forks_count",
        "open_issues_count",
        "language",
        "license",
        "topics",
        "updated_at",
    )
    if k in repo_data
}
print(json.dumps(summary_fields, indent=4))

Pass the repository metadata to Gemini and ask for an executive summary.

In [ ]:
repo_summary_prompt = f"""
You are a technical writer preparing a governance report.
Write a single, concise paragraph (four to six sentences) that serves as an
executive summary of the following GitHub repository. Cover its purpose, primary
audience, current health indicators (stars, forks, open issues), and any notable
governance or licensing details.

Repository metadata (JSON):
{json.dumps(repo_data, indent=2)}
"""

repo_summary_response = client.models.generate_content(
    model=MODEL_ID,
    contents=repo_summary_prompt,
)
print(repo_summary_response.text)

## 2. Analyze file compliance with structured output

Governance pipelines often need to audit source files for common issues: missing license
headers, hardcoded credentials, or use of deprecated libraries. Instead of writing regex
rules for each check, you can ask Gemini to inspect the file and return a structured
compliance report.

Here you fetch the raw content of a notebook from the repository and use
`response_mime_type: "application/json"` together with a `response_schema` to get back
a machine-readable result every time.

In [ ]:
FILE_PATH = "quickstarts/Prompting.ipynb"

raw_response = requests.get(
    f"https://raw.githubusercontent.com/{OWNER}/{REPO}/main/{FILE_PATH}",
    headers=github_headers,
)
raw_response.raise_for_status()
file_content = raw_response.text

print(f"Fetched {FILE_PATH!r} ({len(file_content):,} characters)")

Define the compliance schema and run the structured-output call.

In [ ]:
compliance_schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "has_license_header": types.Schema(
            type=types.Type.BOOLEAN,
            description="True if the file contains an Apache 2.0 license header.",
        ),
        "api_key_hardcoded": types.Schema(
            type=types.Type.BOOLEAN,
            description=(
                "True if a literal API key value (e.g. starting with 'AIzaSy') "
                "appears to be hardcoded in the file."
            ),
        ),
        "uses_deprecated_sdk": types.Schema(
            type=types.Type.BOOLEAN,
            description=(
                "True if the file imports 'google.generativeai' (the deprecated SDK) "
                "instead of 'from google import genai'."
            ),
        ),
        "issues": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(type=types.Type.STRING),
            description=(
                "List of specific compliance problems found. "
                "Empty list if none are found."
            ),
        ),
    },
    required=["has_license_header", "api_key_hardcoded", "uses_deprecated_sdk", "issues"],
)

In [ ]:
compliance_prompt = f"""
You are a compliance auditor for a public open-source repository.
Inspect the following file content and return a JSON compliance report.

Checks to perform:
- has_license_header: does the file contain an Apache 2.0 license header?
- api_key_hardcoded: does the file contain a literal API key (e.g. a string
  starting with 'AIzaSy' or a variable assigned to a literal key string)?
- uses_deprecated_sdk: does the file import 'google.generativeai' (deprecated)
  instead of 'from google import genai'?
- issues: a list of plain-English descriptions of any compliance problems found.

File path: {FILE_PATH}
File content:
---
{file_content[:8000]}
---
"""

compliance_response = client.models.generate_content(
    model=MODEL_ID,
    contents=compliance_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=compliance_schema,
    ),
)

compliance_result = json.loads(compliance_response.text)
print(json.dumps(compliance_result, indent=4))

## 3. PR review assistant

Code reviews are time-consuming. You can use Gemini to give contributors immediate,
actionable feedback on a pull request diff before a human reviewer even looks at it.

The GitHub REST API returns per-file diffs for any pull request at
`/repos/{owner}/{repo}/pulls/{pr_number}/files`. Here you fetch the changed files and
ask Gemini to write a code review comment in the style of a senior engineer.

In [ ]:
# Replace with any open PR number in the target repository.
PR_NUMBER = 1

pr_files_response = requests.get(
    f"https://api.github.com/repos/{OWNER}/{REPO}/pulls/{PR_NUMBER}/files",
    headers=github_headers,
)
pr_files_response.raise_for_status()
pr_files = pr_files_response.json()

print(f"PR #{PR_NUMBER} touches {len(pr_files)} file(s):")
for f in pr_files:
    print(f"  {f['status']:10s}  {f['filename']}")

In [ ]:
# Build a single diff string from all changed files (truncated for context safety).
MAX_DIFF_CHARS = 12000

combined_diff = ""
for f in pr_files:
    combined_diff += f"\n### {f['filename']} ({f['status']})\n"
    combined_diff += f.get("patch", "(binary or no patch available)") + "\n"

combined_diff = combined_diff[:MAX_DIFF_CHARS]

pr_review_prompt = f"""
You are a senior software engineer performing a code review.
Review the following pull request diff and write a clear, actionable review comment.

Your review should:
- Highlight any bugs, security issues, or correctness problems first.
- Call out style or maintainability concerns.
- Acknowledge what is done well.
- Suggest concrete improvements with brief code examples where helpful.
- Be constructive and respectful in tone.

Repository: {OWNER}/{REPO}
PR number: {PR_NUMBER}

Diff:
---
{combined_diff}
---
"""

pr_review_response = client.models.generate_content(
    model=MODEL_ID,
    contents=pr_review_prompt,
)
print(pr_review_response.text)

## 4. Agent schema validator

Projects that govern AI agents through a repository often define each agent as a YAML
file with required metadata fields. Automated validation ensures that new agent
definitions meet the schema before they are merged.

In this section you embed a sample YAML agent definition and ask Gemini to validate it
against a governance schema. Gemini returns a structured compliance report that your
CI/CD pipeline can parse directly.

In [ ]:
# Sample agent definition — as it might appear in a governance repository.
# This example intentionally omits a required field so you can see how
# Gemini surfaces the violation.
sample_agent_yaml = """
id: agent-document-reviewer-v1
name: Document Reviewer
version: "1.2.0"
status: active
audience: internal
# risk_level is intentionally omitted to trigger a compliance violation
description: >
  Reviews legal and compliance documents for missing clauses,
  inconsistent terminology, and regulatory gaps.
capabilities:
  - document_analysis
  - clause_extraction
  - regulatory_mapping
model: gemini-2.5-flash
output_format: structured_json
ip_registry_ref: ip-registry/agents/document-reviewer-v1.json
"""

print(sample_agent_yaml)

Define the validation schema and run the compliance check.

In [ ]:
agent_validation_schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "valid": types.Schema(
            type=types.Type.BOOLEAN,
            description="True if the agent definition passes all governance checks.",
        ),
        "missing_required_fields": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(type=types.Type.STRING),
            description="Names of required fields that are absent from the definition.",
        ),
        "invalid_enum_values": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(type=types.Type.STRING),
            description=(
                "Descriptions of fields whose values fall outside the allowed enum set."
            ),
        ),
        "compliance_report": types.Schema(
            type=types.Type.STRING,
            description=(
                "A plain-English summary of the validation result, suitable for "
                "inclusion in a CI/CD log or PR comment."
            ),
        ),
    },
    required=[
        "valid",
        "missing_required_fields",
        "invalid_enum_values",
        "compliance_report",
    ],
)

In [ ]:
agent_validation_prompt = f"""
You are a governance validator for AI agent definitions.
Validate the following YAML agent definition against the governance schema below
and return a JSON compliance report.

Governance schema rules:
- Required fields: id, name, version, status, audience, risk_level
- 'status' must be one of: draft, active, deprecated, archived
- 'audience' must be one of: internal, external, partner
- 'risk_level' must be one of: low, medium, high, critical
- 'version' must follow semantic versioning (MAJOR.MINOR.PATCH)
- 'id' must be lowercase and use only hyphens as separators

Agent definition (YAML):
---
{sample_agent_yaml}
---
"""

agent_validation_response = client.models.generate_content(
    model=MODEL_ID,
    contents=agent_validation_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=agent_validation_schema,
    ),
)

validation_result = json.loads(agent_validation_response.text)
print(json.dumps(validation_result, indent=4))

## Next steps

### Useful API references

- [Gemini API: Structured output](https://ai.google.dev/gemini-api/docs/structured-output)
- [Gemini API: Text generation](https://ai.google.dev/gemini-api/docs/text-generation)
- [GitHub REST API reference](https://docs.github.com/en/rest)
- [`google-genai` Python SDK reference](https://googleapis.github.io/python-genai/)

### Extend the examples

- Replace the hardcoded `PR_NUMBER` with an input parameter and wire the PR review into a
  GitHub Actions workflow so every pull request gets an automated Gemini review comment.
- Add more fields to the compliance schema in section 2 — for example, checking that
  notebooks use `%pip` rather than `!pip`, or that the model selector uses the Colab
  `# @param` annotation.
- Expand the agent schema validator to load definitions from a directory tree using the
  [GitHub Contents API](https://docs.github.com/en/rest/repos/contents) and report
  aggregate pass/fail counts.

### Continue exploring the Gemini API

- [Entity extraction with JSON](../json_capabilities/Entity_Extraction_JSON.ipynb)
- [PDF structured outputs](../Pdf_structured_outputs_on_invoices_and_forms.ipynb)
- [Search grounding for research reports](../Search_grounding_for_research_report.ipynb)